# Serve the fine-tuned checkpoint from Colab, use it from your laptop

This runs `local_inference_server.py` **unchanged** on a free Colab GPU and exposes it
through a public tunnel, so the workspace on your laptop can select **Local model** and
talk to it over the network.

Why this exists: the checkpoint is a 7B that needed **8.7GB VRAM** to train
(`run_20260822_130636/summary.json`). A 4GB laptop GPU cannot hold it, but Colab's free
T4 has 16GB.

**Before you start:** `Runtime -> Change runtime type -> Hardware accelerator: GPU`.

**What you need to hand:** `adapter.zip` and `local_inference_server.py`, both in
`D:\project blue print\pdsf\`.

**One caveat worth reading:** a Colab session dies after a few hours of use (sooner if
idle), and the tunnel URL is different every time. That is exactly why the workspace has
an editable origin field rather than a hardcoded `127.0.0.1` - paste the new URL in and
press Connect, no restart.

## 1. Confirm you actually got a GPU

If this prints `NO GPU`, fix the runtime type before going further - everything below
assumes CUDA.

In [ ]:
import subprocess, torch

if not torch.cuda.is_available():
    raise SystemExit('NO GPU - Runtime > Change runtime type > GPU, then run this cell again.')

name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU: {name}  ({total_gb:.1f} GB)')
print('bf16 supported:', torch.cuda.is_bf16_supported())
if total_gb < 12:
    print('WARNING: under 12GB. The 4-bit 7B needs roughly 6GB plus headroom.')

## 2. Install the dependencies

Same stack `local_inference_server.py` imports: transformers, peft, bitsandbytes,
accelerate. Takes a couple of minutes.

torch is pinned to Colab's preinstalled version on purpose. Letting pip upgrade it
mid-session breaks `import torch` with a "partially initialized module" error until the
runtime is restarted.

In [ ]:
import importlib.metadata as md

# Pin torch to the version Colab already ships. `pip install -U` would otherwise be free
# to upgrade torch as a dependency, replacing its files on disk while the old torch is
# still loaded in this kernel - the next `import torch` then fails with
# "partially initialized module 'torch' ... circular import".
# Read via package metadata, not `import torch`, so this works even if torch is broken.
torch_version = md.version('torch')
with open('/content/constraints.txt', 'w') as f:
    f.write(f'torch=={torch_version}\n')
print('keeping torch', torch_version)

!pip -q install -U transformers peft bitsandbytes accelerate -c /content/constraints.txt

after = md.version('torch')
print('torch after install:', after)
if after != torch_version:
    print('torch CHANGED - do Runtime > Restart session before running anything else.')
else:
    print('done - torch untouched, no restart needed')

## 3. Upload the adapter and the server script

Both are ready in `D:\project blue print\pdsf\`: upload **`adapter.zip`** and
**`local_inference_server.py`** when prompted (select both at once).

**If you ever rebuild the zip yourself, zip the folder's contents, not the folder:**

```
Compress-Archive -Path 'E:\run_20260822_130636\*' -DestinationPath 'E:\adapter.zip'
```

Zipping the folder itself with Windows PowerShell 5.1 stores entry names with *backslash*
separators (`run_20260822_130636\adapter_config.json`), which the ZIP spec forbids. Linux
does not treat `\` as a separator, so it would extract one flat file literally named that.
The cell below normalises it anyway, so either zip works.

(Uploading ~145MB through the browser is slow. If you would rather not repeat it every
session, put the zip in Google Drive once and use the Drive cell further down instead.)

In [ ]:
from google.colab import files
import zipfile, pathlib

def extract_normalised(zip_path, dest):
    """extractall(), but treating '\\' as a separator - see the note above for why."""
    dest = pathlib.Path(dest).resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            name = info.filename.replace('\\', '/')
            if name.endswith('/'):
                continue
            target = (dest / name).resolve()
            if dest not in target.parents:  # never write outside dest
                raise SystemExit(f'refusing to extract outside {dest}: {name}')
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as src, open(target, 'wb') as out:
                out.write(src.read())

uploaded = files.upload()  # pick adapter.zip AND local_inference_server.py

for filename in uploaded:
    if filename.endswith('.zip'):
        extract_normalised(filename, '/content/adapter_root')
        print('extracted', filename)

candidates = list(pathlib.Path('/content/adapter_root').rglob('adapter_config.json'))
if not candidates:
    raise SystemExit('No adapter_config.json found in the zip - is this the right folder?')
ADAPTER_DIR = str(candidates[0].parent)
print('ADAPTER_DIR =', ADAPTER_DIR)

### Alternative: load the adapter from Google Drive

Skip the cell above and use this one instead if you have put `adapter.zip` in your Drive -
no re-upload each session.

In [ ]:
# Needs extract_normalised from the upload cell - run that cell's def first (skip its upload).
# from google.colab import drive
# import pathlib
# drive.mount('/content/drive')
# extract_normalised('/content/drive/MyDrive/adapter.zip', '/content/adapter_root')
# ADAPTER_DIR = str(list(pathlib.Path('/content/adapter_root').rglob('adapter_config.json'))[0].parent)
# print('ADAPTER_DIR =', ADAPTER_DIR)

## 4. Make the script T4-safe

`local_inference_server.py` asks bitsandbytes for `bnb_4bit_compute_dtype=torch.bfloat16`.
Native bfloat16 needs an Ampere GPU (compute capability 8.0+). Colab's free tier usually
hands out a **T4**, which is Turing (7.5).

Note that `torch.cuda.is_bf16_supported()` can print **True** on a T4 with recent PyTorch,
because it counts software emulation. That's why cell 1 may say `bf16 supported: True`.
This cell decides from the compute capability instead, so a T4 still gets patched.

It rewrites exactly that one dtype and leaves the rest of your file - including the JSON
punctuation-constrained decoding - untouched.

In [ ]:
import pathlib, torch

script = pathlib.Path('/content/local_inference_server.py')
source = script.read_text(encoding='utf-8')

# Decide by compute capability, NOT torch.cuda.is_bf16_supported(): newer PyTorch counts
# software EMULATION as support by default, so a T4 reports True even though bf16 on it
# is emulated and very slow. Native bf16 needs Ampere (compute capability 8.0) or newer.
major, minor = torch.cuda.get_device_capability(0)
native_bf16 = major >= 8
print(f'compute capability {major}.{minor} -> native bf16: {native_bf16}')

if native_bf16:
    print('Native bf16 on this GPU - leaving the script exactly as written.')
else:
    patched = source.replace('bnb_4bit_compute_dtype=torch.bfloat16',
                             'bnb_4bit_compute_dtype=torch.float16')
    if patched == source:
        print('WARNING: expected dtype line not found - check the script manually.')
    else:
        script.write_text(patched, encoding='utf-8')
        print('Patched bfloat16 -> float16 for this non-Ampere GPU.')

## 5. Start the server and open a public tunnel

`cloudflared` quick tunnels need no account and no auth token.

The first run downloads Qwen2.5-7B-Instruct (~15GB) from HuggingFace, which takes a few
minutes. Loading is done when you see `READY: serving POST /complete`.

Your server binds to `127.0.0.1`, and cloudflared runs on this same machine, so it reaches
it fine - nothing in the script needs changing for the tunnel to work.

In [ ]:
import subprocess, threading, time, re, os

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

PORT = 8712
MODEL_NAME = 'local:qwen2.5-7b-instruct+run_20260822_130636'

server = subprocess.Popen(
    ['python', '/content/local_inference_server.py',
     '--adapter', ADAPTER_DIR, '--port', str(PORT), '--model-name', MODEL_NAME],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

def echo(process, tag):
    for line in process.stdout:
        print(f'[{tag}] {line}', end='')

threading.Thread(target=echo, args=(server, 'server'), daemon=True).start()

print('Loading the model (first run downloads ~15GB) - waiting for READY...')
deadline = time.time() + 1800
while time.time() < deadline:
    if server.poll() is not None:
        raise SystemExit('The server exited early - read the [server] output above.')
    try:
        import socket
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            break
    except OSError:
        time.sleep(5)
else:
    raise SystemExit('Server did not come up within 30 minutes.')

print('\nServer is listening. Opening the tunnel...\n')

tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public_url = None
for line in tunnel.stdout:
    print('[tunnel]', line, end='')
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if found:
        public_url = found.group(0)
        break

threading.Thread(target=echo, args=(tunnel, 'tunnel'), daemon=True).start()

print('\n' + '=' * 64)
print('PASTE THIS INTO THE WORKSPACE (Model -> Local model -> origin field):')
print('   ', public_url)
print('=' * 64)

## 6. Prove it works before switching over

This sends a real request in exactly the shape `src/llm/local.ts` sends, and checks the
reply matches the `CompletionResult` contract. If this passes, the workspace will work.

In [ ]:
import json, urllib.request

payload = {
    'system': 'You reply with one JSON object and nothing else.',
    'user': 'Reply with {"ok": true}',
    'maxOutputTokens': 64,
}
request = urllib.request.Request(
    f'http://127.0.0.1:{PORT}/complete',
    data=json.dumps(payload).encode(),
    headers={'content-type': 'application/json'},
)
with urllib.request.urlopen(request, timeout=300) as response:
    result = json.load(response)

print(json.dumps(result, indent=2)[:900])
assert isinstance(result.get('ok'), bool), 'reply is missing the boolean `ok` local.ts requires'
if result['ok']:
    latency = result.get('_debug', {}).get('latencySeconds')
    print(f'\nOK - generation works. {latency:.1f}s for this request.' if latency else '\nOK.')
else:
    print('\nThe server answered but reported a failure - see `error` above.')

## 7. Keep this tab open

The tunnel and the server live only as long as this runtime. Closing the tab, or letting
Colab time out, kills both - and the next session gets a **different URL**, which you paste
into the workspace again.

### If generation times out in the workspace

`src/llm/local.ts` gives up after 300s. A T4 serving a 7B is far slower than Gemini, and
component code generation asks for up to 4096 tokens. If you see `network error:
TimeoutError`, raise `REQUEST_TIMEOUT_MS` in that file - the request is working, it is
just slower than the ceiling allows.